# Machine Learning Crash Course: Beginner to Advanced
**David Oluyale | Agentic AI Hub**

This notebook takes you from "what is machine learning?" to training, checking, and saving real models, in about an hour.
Every idea uses a small, everyday example: house prices, exam results, fruit, spam, loans, shop customers.

The tables in `data/` are **made-up example data, for teaching**. No API keys and no setup. Just run each cell from top to bottom.

**What you will learn**
1. How machine learning works: features, labels, train and test.
2. Linear regression, and how to measure error (MAE, RMSE, R²).
3. Logistic regression, and probabilities.
4. KNN, Naive Bayes and SVM, side by side.
5. Decision trees and random forests.
6. Gradient boosting.
7. Data prep and feature engineering.
8. Honest evaluation: the confusion matrix, precision, recall, ROC and imbalanced data.
9. Overfitting, the bias–variance tradeoff, regularisation and cross-validation.
10. Pipelines and GridSearch for hyperparameter tuning.
11. Unsupervised learning: k-means and PCA.
12. A small neural network, on handwritten digits.
13. Saving a model with joblib, and where to go next.

## Part 1. How machine learning works
Ordinary software follows rules we write. **Machine learning** is different: we show the computer examples, and it finds a useful pattern.

A **feature** is an input we already know. A **label** is the answer we want to predict. Together, they make a **labelled example**.

In [ ]:
# SETUP: helper code (shown in full). Loads each example table, or builds it here if the download fails.
from pathlib import Path
from io import StringIO
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({
    "figure.figsize": (8, 4.4),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.titlesize": 14,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})
TEAL, CORAL, INK, PURPLE = "#0F766E", "#EA580C", "#1F2937", "#4F46E5"

RAW = "https://raw.githubusercontent.com/davolu/machine-learning-crash-course/main/data"
FALLBACK = {
    "churn.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
months,monthly_bill,support_calls,churned
10,93,0,0
29,99,1,0
16,42,2,0
45,92,1,0
33,44,0,0
33,62,2,0
48,83,1,0
35,68,0,0
10,76,0,0
52,97,1,0
14,91,0,0
9,107,2,0
42,93,0,0
25,70,0,0
35,46,1,0
55,90,2,0
28,59,0,0
10,70,0,0
48,60,4,0
54,90,3,0
36,55,5,0
60,70,4,0
17,51,6,1
17,68,4,1
20,96,7,1
16,53,7,1
23,99,3,1
11,105,7,1
16,108,7,1
14,116,4,1
9,98,6,1
5,110,3,1
29,40,6,1
28,106,6,1
17,99,4,1
28,87,5,1
18,88,3,1
5,60,3,1
16,96,6,1
17,112,5,1
2,120,1,1
3,110,0,1
4,95,2,1
1,130,1,1
""",
    "customers.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
age,annual_spend,store_visits
21,337,4
28,282,3
27,205,3
25,347,5
25,270,6
29,280,3
21,254,6
27,216,6
23,365,4
21,336,5
26,308,3
30,260,6
28,344,5
28,289,4
28,268,3
50,1607,7
43,1509,6
49,1301,6
46,1632,7
48,1325,8
47,1279,9
40,1598,7
42,1303,5
44,1141,9
44,1358,8
38,1537,8
45,1204,5
40,1354,6
47,1171,8
46,1477,9
65,646,16
70,645,18
70,616,17
65,567,16
71,626,19
63,519,17
63,645,14
68,588,18
68,564,17
61,576,17
70,592,17
62,459,17
70,484,14
60,511,17
70,478,18
""",
    "fraud.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
amount,hour,international,is_fraud
85,12,0,0
117,7,0,0
76,18,0,0
27,11,0,0
73,19,0,0
27,14,1,0
113,16,0,0
35,22,0,0
48,8,0,0
88,8,0,0
177,22,0,0
29,13,0,0
64,13,0,0
102,12,0,0
11,10,0,0
81,12,0,0
113,7,0,0
73,18,0,0
5,18,0,0
38,7,0,0
26,13,0,0
53,16,0,0
105,10,1,0
46,9,0,0
6,9,1,0
94,10,0,0
48,18,0,0
54,19,0,0
51,7,0,0
94,8,0,0
94,17,0,0
65,10,0,0
5,7,0,0
20,20,0,0
51,22,0,0
106,13,0,0
22,20,0,0
5,10,0,0
40,19,0,0
43,12,0,0
150,16,0,0
78,3,0,0
111,7,0,0
10,8,0,0
50,11,0,0
58,22,0,0
85,6,0,0
215,21,0,0
5,10,0,0
49,12,1,0
123,20,0,0
18,7,0,0
67,17,0,0
119,20,0,0
21,14,0,0
28,19,0,0
116,10,0,0
67,21,0,0
47,12,0,0
78,18,0,0
64,14,0,0
80,16,0,0
91,18,0,0
8,9,0,0
156,8,0,0
143,20,0,0
25,18,0,0
69,7,1,0
140,7,0,0
5,14,0,0
9,8,0,0
115,17,0,0
45,10,0,0
96,22,0,0
35,2,0,0
78,7,0,0
81,22,0,0
27,19,0,0
40,1,0,0
33,18,0,0
61,15,0,0
180,12,0,0
126,16,0,0
5,9,0,0
15,19,0,0
11,15,0,0
156,10,0,0
108,8,0,0
63,7,1,0
56,8,1,0
93,16,0,0
87,14,0,0
33,3,0,0
210,9,0,0
89,3,0,0
83,7,0,0
83,21,0,0
116,20,0,0
46,18,0,0
55,7,0,0
77,19,0,0
58,16,0,0
51,13,0,0
156,11,0,0
53,11,0,0
53,22,0,0
156,11,0,0
157,16,0,0
55,16,0,0
119,8,0,0
60,15,0,0
161,6,0,0
66,22,0,0
145,18,0,0
19,9,0,0
36,15,0,0
127,15,0,0
85,18,0,0
150,15,0,0
99,7,0,0
88,12,0,0
110,11,0,0
81,10,0,0
62,8,0,0
119,12,0,0
144,14,0,0
80,13,0,0
136,17,0,0
97,19,0,0
5,15,0,0
75,13,0,0
7,13,0,0
91,20,0,0
5,23,0,0
46,19,0,0
76,7,0,0
96,9,0,0
79,8,0,0
96,2,1,0
111,15,0,0
5,19,1,0
107,21,0,0
124,19,1,0
31,7,0,0
137,16,0,0
40,17,0,0
52,17,0,0
5,4,0,0
93,20,0,0
116,7,0,0
16,13,0,0
60,15,0,0
136,10,0,0
123,16,0,0
89,14,0,0
167,7,0,0
78,10,0,0
86,16,0,0
28,9,0,0
14,2,0,0
166,9,0,0
87,17,0,0
79,10,0,0
123,12,0,0
199,14,0,0
74,21,0,0
61,18,0,0
27,13,0,0
103,19,1,0
55,11,0,0
116,18,1,0
5,19,0,0
114,15,0,0
5,22,0,0
91,10,0,0
5,12,0,0
65,9,0,0
103,8,0,0
151,14,0,0
136,7,0,0
820,2,1,1
90,3,1,1
310,23,1,1
540,1,0,1
45,4,1,1
700,0,1,1
120,22,0,1
880,2,1,1
150,14,1,1
610,5,0,1
990,1,1,1
70,23,1,1
260,3,0,1
40,2,1,1
430,19,1,1
75,11,1,1
520,4,0,1
200,0,1,1
35,5,0,1
340,22,1,1
""",
    "fruit.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
weight_g,colour_score,fruit
148,0.12,apple
162,0.18,apple
171,0.08,apple
155,0.22,apple
180,0.15,apple
140,0.25,apple
175,0.1,apple
158,0.2,apple
168,0.05,apple
152,0.28,apple
130,0.82,orange
145,0.9,orange
138,0.75,orange
160,0.88,orange
125,0.95,orange
152,0.8,orange
142,0.7,orange
135,0.92,orange
148,0.85,orange
158,0.78,orange
""",
    "houses.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
size_sqm,bedrooms,price_k
177,4,484.7
134,3,383.9
142,1,395.9
171,3,482.0
128,3,365.3
154,3,418.8
162,2,418.7
80,1,272.6
57,1,173.6
90,1,246.1
88,3,299.0
167,3,452.6
173,3,463.2
50,2,161.0
117,2,330.4
160,3,429.5
67,1,236.0
157,3,451.7
66,1,203.0
113,2,339.7
160,4,452.2
90,1,281.4
96,3,306.5
87,2,282.9
147,2,360.1
84,3,290.4
183,4,465.4
110,1,243.1
114,2,316.5
118,3,329.4
128,2,358.1
124,3,423.8
118,2,311.2
184,4,495.2
159,4,464.2
157,3,447.5
144,3,402.8
133,2,358.9
96,2,304.9
183,3,502.8
112,2,293.3
79,1,228.8
164,4,470.3
71,2,206.7
165,2,438.3
132,2,339.9
65,2,246.8
55,1,185.5
""",
    "loans.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
income_k,credit_score,years_at_job,approved
72,760,6,1
95,810,10,1
60,740,5,1
110,790,8,1
80,720,7,1
55,750,4,1
88,770,9,1
100,800,3,1
67,730,8,1
74,745,5,1
90,785,6,1
82,755,11,1
28,540,1,0
32,580,0,0
24,510,2,0
40,600,1,0
35,560,0,0
22,530,1,0
30,590,2,0
38,610,0,0
26,500,3,0
33,570,1,0
29,550,0,0
42,620,2,0
48,680,2,1
50,650,1,0
52,700,3,1
45,640,4,0
58,690,1,1
44,660,2,0
62,630,6,1
47,670,0,0
53,645,5,0
70,680,2,1
41,700,1,0
64,655,3,1
49,615,4,0
77,640,8,1
36,690,5,0
85,610,7,0
""",
    "spam.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
text,is_spam
"win a free prize now, click this link",1
"congratulations you won money, claim your cash prize",1
"urgent: your account will be closed, verify here for a free gift",1
"cheap meds and a free holiday, limited offer winner",1
"you have been selected to win a lucky prize, act now",1
"claim your reward, 100 percent free, no credit card",1
"winner winner, cash prize waiting, click to collect",1
"exclusive deal, make money fast from home today",1
"hi, are we still meeting for lunch tomorrow?",0
please find the project notes attached,0
thanks for your help with the report yesterday,0
can you send the invoice for the office chairs?,0
"the train is late, I will be there at 10",0
"happy birthday, see you at the party on Saturday",0
here is the agenda for Monday's team meeting,0
I booked a table at the new restaurant,0
""",
    "students.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
hours,attendance_pct,passed
1.0,42,0
1.5,55,0
2.0,48,0
2.5,60,0
3.0,51,0
3.5,70,0
4.0,58,0
4.5,64,0
5.0,50,0
5.5,72,1
6.0,61,0
6.0,80,1
6.5,74,1
7.0,55,0
7.0,82,1
7.5,70,1
8.0,66,1
8.5,88,1
9.0,60,1
9.0,91,1
9.5,77,1
10.0,68,1
10.5,85,1
11.0,79,1
11.5,94,1
2.0,88,0
3.0,90,0
12.0,45,0
12.0,92,1
8.0,48,0
""",
    "used_cars.csv": r"""# Example data for the Machine Learning Crash Course. Made-up, for teaching.
year,mileage,brand,fuel,accidents,price_k
2018,42000.0,Toyota,petrol,0.0,14.8
2015,78000.0,Honda,petrol,1.0,9.6
2020,18000.0,Toyota,hybrid,0.0,22.4
2012,,Ford,diesel,2.0,5.1
2019,31000.0,BMW,petrol,0.0,24.0
2016,65000.0,Honda,diesel,,10.2
2021,9000.0,Toyota,hybrid,0.0,26.5
2014,92000.0,Ford,,1.0,6.4
2017,54000.0,BMW,petrol,0.0,18.7
2013,110000.0,Honda,petrol,3.0,4.8
2022,4000.0,BMW,hybrid,0.0,32.0
2011,130000.0,Ford,diesel,2.0,3.9
2018,,Honda,petrol,0.0,13.5
2016,70000.0,Toyota,petrol,1.0,11.1
2019,25000.0,Ford,diesel,,12.8
2014,88000.0,BMW,petrol,1.0,11.9
"""
}

def load_csv(name):
    """Load a made-up example table. Tries local data/, then GitHub, then a built-in copy."""
    local = Path("data") / name
    text = None
    if local.exists():
        text = local.read_text()
    else:
        try:
            with urllib.request.urlopen(f"{RAW}/{name}", timeout=8) as r:
                text = r.read().decode()
            Path("data").mkdir(parents=True, exist_ok=True)
            local.write_text(text)
        except Exception:
            text = FALLBACK[name]
            Path("data").mkdir(parents=True, exist_ok=True)
            local.write_text(text)
            print(f"GitHub was not available, so {name} was built from the built-in example table.")
    df = pd.read_csv(StringIO(text), comment="#")
    print(f"Loaded {name} — a small made-up example table, for practice. {len(df)} rows.")
    return df

print("Ready. numpy, pandas and matplotlib are loaded.")

Here is a tiny table of students. `hours` and `attendance_pct` are **features**. `passed` is the **label** (1 means pass, 0 means fail).

In [ ]:
students = load_csv("students.csv")
print(students.head(8).to_string(index=False))
print()
print("Features:", ["hours", "attendance_pct"])
print("Label:   ", "passed")

**Supervised learning** uses labels. We already know who passed, so we can teach a model to predict the next student.

**Unsupervised learning** has no labels. We only have measurements, and we ask the computer to find groups or patterns.

In [ ]:
print("Supervised example:  predict passed  from  hours + attendance")
print("                     (we know the answers while training)")
print()
print("Unsupervised example: group shop customers by age, spend and visits")
print("                     (there is no pre-written 'group' column)")

We never test a model on the same rows it trained on. That would be like giving a student the exam questions in advance.

So we **split** the table: most rows for **training**, a held-out slice for **testing**. `random_state=42` makes the split the same every time you run it.

In [ ]:
from sklearn.model_selection import train_test_split

X = students[["hours", "attendance_pct"]]
y = students["passed"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows: ", len(X_test))
print("Passes in train:", int(y_train.sum()), "  passes in test:", int(y_test.sum()))

## Part 2. Linear regression
**Linear regression** draws a straight line (or a flat surface, with more than one feature) through numbered data.

We will predict a house price from its size and the number of bedrooms. `price_k` is the price in thousands.

In [ ]:
houses = load_csv("houses.csv")
print(houses.head(6).to_string(index=False))
print()
print(houses.describe().round(1).to_string())

A scatter plot is the fastest way to see a pattern. If prices rise as houses get bigger, a straight line is a fair first model.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(houses["size_sqm"], houses["price_k"], c=TEAL, s=50, edgecolors="white", linewidths=0.6)
ax.set_xlabel("Size (square metres)")
ax.set_ylabel("Price (thousands)")
ax.set_title("House prices vs size")
plt.show()

`LinearRegression` finds the slope and the intercept that keep the errors as small as possible. We fit on the training rows only, then try a new house: 100 m², 3 bedrooms.

In [ ]:
from sklearn.linear_model import LinearRegression

X = houses[["size_sqm", "bedrooms"]]
y = houses["price_k"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

linreg = LinearRegression()
linreg.fit(X_train, y_train)

print("Slope for size_sqm: ", round(linreg.coef_[0], 2))
print("Slope for bedrooms: ", round(linreg.coef_[1], 2))
print("Intercept:          ", round(linreg.intercept_, 1))

new_house = pd.DataFrame({"size_sqm": [100], "bedrooms": [3]})
print("Predicted price for 100 m², 3 bedrooms:", round(linreg.predict(new_house)[0], 1), "thousand")

A prediction is only useful if we know how wrong it tends to be. Three common scores:

- **MAE**: the average miss, in the same units as the price.
- **RMSE**: like MAE, but bigger misses count for more.
- **R²**: 1 is a perfect fit, 0 means "no better than predicting the average".

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = linreg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
print(f"MAE:  {mae:.1f}  (average miss, in thousands)")
print(f"RMSE: {rmse:.1f}")
print(f"R²:   {r2:.3f}")

fig, ax = plt.subplots()
ax.scatter(X_test["size_sqm"], y_test, c=TEAL, s=60, label="Actual", zorder=3)
ax.scatter(X_test["size_sqm"], y_pred, c=CORAL, s=60, marker="D", label="Predicted", zorder=3)
for x, a, p in zip(X_test["size_sqm"], y_test, y_pred):
    ax.plot([x, x], [a, p], color="grey", linewidth=1, alpha=0.6)
ax.set_xlabel("Size (square metres)")
ax.set_ylabel("Price (thousands)")
ax.set_title("Test houses: actual vs predicted")
ax.legend()
plt.show()

## Part 3. Logistic regression
When the answer is a category, such as pass or fail, we use **classification**.

**Logistic regression** is a simple, strong starting point. It does not jump straight to a yes or no. It first estimates a **probability** between 0 and 1, then uses a threshold (usually 0.5).

In [ ]:
from sklearn.linear_model import LogisticRegression

X = students[["hours", "attendance_pct"]]
y = students["passed"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
print("Test accuracy:", round(logreg.score(X_test, y_test), 3))
print("Hours weight:      ", round(logreg.coef_[0, 0], 3))
print("Attendance weight: ", round(logreg.coef_[0, 1], 3))

Accuracy is a start, but the probability is the interesting part. Let's try three made-up students: one who barely studied, one in the middle, and one who worked hard.

In [ ]:
demo = pd.DataFrame({
    "hours": [2, 6, 10],
    "attendance_pct": [50, 70, 90],
})
preds = logreg.predict(demo)
proba = logreg.predict_proba(demo)[:, 1]
demo["pass_probability"] = np.round(proba, 3)
demo["predicted"] = np.where(preds == 1, "pass", "fail")
print(demo.to_string(index=False))

If we hold attendance at 70% and slide hours from 0 to 12, the probability should rise in a smooth S-shape. That curve is why this model is called "logistic".

In [ ]:
hours = np.linspace(0, 12, 80)
grid = pd.DataFrame({"hours": hours, "attendance_pct": 70})
curve = logreg.predict_proba(grid)[:, 1]

fig, ax = plt.subplots()
ax.plot(hours, curve, color=TEAL, linewidth=3)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, label="0.5 threshold")
ax.scatter(demo["hours"], proba, c=CORAL, s=80, zorder=3, label="Our three students")
ax.set_xlabel("Hours studied (attendance held at 70%)")
ax.set_ylabel("Probability of passing")
ax.set_title("Logistic regression gives a smooth probability")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.show()

## Part 4. KNN, Naive Bayes and SVM
Three classic classifiers, each with a different everyday example.

**K-nearest neighbours (KNN)** looks at the closest training examples and takes a vote. We will sort fruit into apples and oranges using weight and a colour score (0 is redder, 1 is more orange).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

fruit = load_csv("fruit.csv")
X = fruit[["weight_g", "colour_score"]]
y = fruit["fruit"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
print("Test accuracy:", round(knn.score(X_test, y_test), 3))

new_fruit = pd.DataFrame({"weight_g": [150, 140], "colour_score": [0.15, 0.85]})
print(new_fruit.assign(predicted=knn.predict(new_fruit)).to_string(index=False))

fig, ax = plt.subplots()
for name, col in [("apple", CORAL), ("orange", "#F59E0B")]:
    part = fruit[fruit["fruit"] == name]
    ax.scatter(part["weight_g"], part["colour_score"], c=col, s=70, label=name, edgecolors="white")
ax.scatter(new_fruit["weight_g"], new_fruit["colour_score"], c=INK, s=90, marker="X", label="new fruit", zorder=3)
ax.set_xlabel("Weight (grams)")
ax.set_ylabel("Colour score (0 = redder, 1 = more orange)")
ax.set_title("Fruit: apples and oranges")
ax.legend()
plt.show()

**Naive Bayes** is great for text. It asks, "Given these words, how likely is this message to be spam?" We turn each message into word counts with `CountVectorizer`, then fit a `MultinomialNB` model.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

spam = load_csv("spam.csv")
X_train, X_test, y_train, y_test = train_test_split(
    spam["text"], spam["is_spam"], test_size=0.31, random_state=42, stratify=spam["is_spam"]
)
vectoriser = CountVectorizer()
X_train_counts = vectoriser.fit_transform(X_train)
X_test_counts = vectoriser.transform(X_test)

nb = MultinomialNB()
nb.fit(X_train_counts, y_train)
print("Test accuracy:", round(accuracy_score(y_test, nb.predict(X_test_counts)), 3))

messages = [
    "click now to win a free cash prize",
    "shall we meet for coffee tomorrow?",
]
probs = nb.predict_proba(vectoriser.transform(messages))[:, 1]
for msg, pred, p in zip(messages, nb.predict(vectoriser.transform(messages)), probs):
    label = "spam" if pred == 1 else "not spam"
    print(f"{p:.3f}  {label:8}  {msg}")

**Support Vector Machines (SVM)** look for the widest possible gap between the classes. We will use two measurements from the famous Iris flowers: petal length and petal width. This is a built-in teaching set, not our made-up tables.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC

iris = load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]]
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
svm = SVC(kernel="rbf", C=1.0, random_state=42)
svm.fit(X_train, y_train)
print("Classes:", ", ".join(str(n) for n in iris.target_names))
print("Test accuracy:", round(svm.score(X_test, y_test), 3))

How do the three models compare, on the same flowers? We fit each one on the same split and print a small score table. A close result is common. In practice, we also care about speed, simplicity, and how well the method fits the data.

In [ ]:
from sklearn.naive_bayes import GaussianNB

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", C=1.0, random_state=42),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append((name, round(model.score(X_test, y_test), 3)))
print(pd.DataFrame(rows, columns=["model", "test_accuracy"]).to_string(index=False))

## Part 5. Decision trees and random forests
A **decision tree** asks a sequence of yes/no questions, such as "is income above 50?" Each answer sends you down a branch, until you reach a decision.

We will predict whether a loan is approved from income, credit score, and years in the job.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

loans = load_csv("loans.csv")
X = loans[["income_k", "credit_score", "years_at_job"]]
y = loans["approved"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print("Train accuracy:", round(tree.score(X_train, y_train), 3))
print("Test accuracy: ", round(tree.score(X_test, y_test), 3))
print()
print(export_text(tree, feature_names=list(X.columns)))

The same tree, drawn as a picture. Each box is a question or a final decision. We keep it shallow (`max_depth=3`) so you can read every step.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.2))
plot_tree(
    tree, feature_names=list(X.columns), class_names=["declined", "approved"],
    filled=True, rounded=True, impurity=False, ax=ax, fontsize=9,
)
ax.set_title("A shallow loan-approval tree")
plt.show()

A single tree can **memorise** the training rows if we let it grow deep. A **random forest** grows many trees, each on a slightly different slice of the data, and lets them vote. That usually generalises better.

**Feature importance** tells us which inputs the forest relied on most.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

deep = DecisionTreeClassifier(random_state=42)
deep.fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=80, random_state=42)
forest.fit(X_train, y_train)

print(f"{'model':<22} {'train':>7} {'test':>7}")
print(f"{'Shallow tree (depth 3)':<22} {tree.score(X_train, y_train):7.3f} {tree.score(X_test, y_test):7.3f}")
print(f"{'Deep tree (no limit)':<22} {deep.score(X_train, y_train):7.3f} {deep.score(X_test, y_test):7.3f}")
print(f"{'Random forest':<22} {forest.score(X_train, y_train):7.3f} {forest.score(X_test, y_test):7.3f}")
print("Deep tree depth:", deep.get_depth())

imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.6))
imp.plot(kind="barh", ax=ax, color=TEAL)
ax.set_xlabel("Importance")
ax.set_title("What the forest paid attention to")
plt.show()

## Part 6. Gradient boosting
**Gradient boosting** also combines many trees, but it builds them **in sequence**. Each new tree tries to fix the mistakes of the ones before it.

We will predict **customer churn**: who will cancel a subscription, using months with us, the monthly bill, and how many times they called support.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

churn = load_csv("churn.csv")
X = churn[["months", "monthly_bill", "support_calls"]]
y = churn["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
boost = GradientBoostingClassifier(random_state=42)
boost.fit(X_train, y_train)
print("Train accuracy:", round(boost.score(X_train, y_train), 3))
print("Test accuracy: ", round(boost.score(X_test, y_test), 3))

imp = pd.Series(boost.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.6))
imp.plot(kind="barh", ax=ax, color=CORAL)
ax.set_xlabel("Importance")
ax.set_title("What the boosting model paid attention to")
plt.show()

Try two customers. One has been with us for years, pays a modest bill, and almost never calls. The other is new, pays a high bill, and has called six times.

In [ ]:
two = pd.DataFrame({
    "months": [40, 3],
    "monthly_bill": [45, 100],
    "support_calls": [0, 6],
})
churn_preds = boost.predict(two)
churn_proba = boost.predict_proba(two)[:, 1]
two["churn_probability"] = np.round(churn_proba, 3)
two["predicted"] = np.where(churn_preds == 1, "churn", "stay")
print(two.to_string(index=False))

## Part 7. Data prep and feature engineering
Real tables are messy. Values go missing, numbers come in different scales, and some columns are words, not numbers.

This used-car table has all three problems. We will clean it, then **engineer** new features that a model can use.

In [ ]:
cars = load_csv("used_cars.csv")
print(cars.to_string(index=False))
print()
print("Missing values in each column:")
print(cars.isna().sum().to_string())

For missing **numbers**, a simple start is the median. For missing **categories**, we use the most common value (the mode). `SimpleImputer` learns those values from the data, then fills the gaps.

In [ ]:
from sklearn.impute import SimpleImputer

cars_clean = cars.copy()
num_cols = ["mileage", "accidents"]
cat_cols = ["fuel"]

num_imputer = SimpleImputer(strategy="median")
cars_clean[num_cols] = num_imputer.fit_transform(cars_clean[num_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
cars_clean[cat_cols] = cat_imputer.fit_transform(cars_clean[cat_cols])
print(cars_clean.to_string(index=False))

**Scaling** puts numeric columns on a similar footing. Without it, a mileage of 110,000 would swamp a column like `accidents`, which is only 0, 1 or 2. `StandardScaler` turns each column into "how many steps above or below the average".

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(cars_clean[["year", "mileage", "accidents", "price_k"]])
scaled_df = pd.DataFrame(scaled, columns=["year", "mileage", "accidents", "price_k"]).round(2)
print("Means after scaling (should be about 0):", np.round(scaled.mean(axis=0), 3))
print()
print(scaled_df.head().to_string(index=False))

**Encoding** turns categories into numbers. **One-hot encoding** makes a 0/1 column for each brand and each fuel type, so the model never has to pretend that "Toyota" is greater than "Ford".

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(cars_clean[["brand", "fuel"]])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(), index=cars_clean.index)
print(encoded_df.head().to_string(index=False))

**Feature engineering** is inventing better inputs from the ones we have. A car's age is clearer than its year. Miles per year tells us how hard the car was used. The year of this course is 2026.

In [ ]:
cars_feat = cars_clean.copy()
cars_feat["age_years"] = 2026 - cars_feat["year"]
cars_feat["miles_per_year"] = (cars_feat["mileage"] / cars_feat["age_years"]).round(0)
print(cars_feat[["year", "age_years", "mileage", "miles_per_year", "brand", "price_k"]].to_string(index=False))

## Part 8. Evaluation
Accuracy can lie. Imagine a fraud detector that always says "not fraud". If only 1 in 10 payments is fraud, that "model" is 90% accurate — and completely useless.

This payments table is **imbalanced**: most rows are genuine.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score,
)

fraud = load_csv("fraud.csv")
print(fraud["is_fraud"].value_counts().to_string())
print("Fraud rate:", f"{fraud['is_fraud'].mean():.1%}")

X = fraud[["amount", "hour", "international"]]
y = fraud["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Fraud cases in the test set:", int(y_test.sum()), "of", len(y_test))

A **dummy** model always predicts "not fraud". Then we fit a plain logistic regression and compare **accuracy, precision, recall and F1**.

- **Precision**: of the rows we flagged as fraud, how many really were?
- **Recall**: of the real fraud, how many did we catch?
- **F1**: a balance of the two.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
plain = LogisticRegression(max_iter=1000, random_state=42)
plain.fit(X_train, y_train)

def report(name, model):
    pred = model.predict(X_test)
    rec = dict(
        model=name,
        accuracy=round(accuracy_score(y_test, pred), 3),
        precision=round(precision_score(y_test, pred, zero_division=0), 3),
        recall=round(recall_score(y_test, pred, zero_division=0), 3),
        f1=round(f1_score(y_test, pred, zero_division=0), 3),
    )
    return rec, pred

dummy_row, dummy_pred = report("always not-fraud", dummy)
plain_row, plain_pred = report("plain logistic", plain)
print(pd.DataFrame([dummy_row, plain_row]).to_string(index=False))
print()
print("Confusion matrix for plain logistic  [true 0 / true 1 as rows]:")
print(confusion_matrix(y_test, plain_pred))

The **confusion matrix** makes those numbers visual. The bottom-left cell is fraud we missed. The top-right cell is genuine payments we wrongly flagged.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.6))
ConfusionMatrixDisplay.from_predictions(
    y_test, plain_pred, display_labels=["genuine", "fraud"], cmap="Greens", ax=ax, colorbar=False,
)
ax.set_title("Plain logistic regression")
plt.show()

A **ROC curve** plots the true-positive rate against the false-positive rate as we slide the threshold. **AUC** is the area under that curve: 0.5 is a coin flip, 1.0 is perfect.

`class_weight="balanced"` tells the model that missing a rare fraud is costly, so it tries harder to catch those rows.

In [ ]:
balanced = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
balanced.fit(X_train, y_train)
bal_row, bal_pred = report("balanced logistic", balanced)
print(pd.DataFrame([dummy_row, plain_row, bal_row]).to_string(index=False))
print("AUC plain:    ", round(roc_auc_score(y_test, plain.predict_proba(X_test)[:, 1]), 3))
print("AUC balanced: ", round(roc_auc_score(y_test, balanced.predict_proba(X_test)[:, 1]), 3))

fig, ax = plt.subplots()
RocCurveDisplay.from_estimator(plain, X_test, y_test, ax=ax, name="plain")
RocCurveDisplay.from_estimator(balanced, X_test, y_test, ax=ax, name="balanced")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1, label="coin flip")
ax.set_title("ROC curves")
ax.legend()
plt.show()

## Part 9. Overfitting, regularisation and cross-validation
A model **underfits** when it is too simple to catch the pattern. It **overfits** when it memorises the training noise and then stumbles on new data.

The sweet spot sits in the middle. That is the **bias–variance tradeoff**: simple models can be biased, very flexible models can vary wildly.

Here is a wavy pattern with noise, like a shop's footfall through the day.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = np.sin(x) + 0.15 * x + rng.normal(0, 0.45, size=x.shape)
X = x.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

fig, ax = plt.subplots()
ax.scatter(X_train[:, 0], y_train, c=TEAL, s=55, label="train", zorder=3)
ax.scatter(X_test[:, 0], y_test, c=CORAL, s=55, marker="D", label="test", zorder=3)
curve_x = np.linspace(0, 10, 200)
ax.plot(curve_x, np.sin(curve_x) + 0.15 * curve_x, color=INK, linewidth=1.5, alpha=0.7, label="true pattern")
ax.set_xlabel("Time of day (made-up scale)")
ax.set_ylabel("Footfall (made-up scale)")
ax.set_title("A wavy pattern, plus noise")
ax.legend()
plt.show()

We will grow decision trees of increasing depth. Watch the **training** error fall towards zero, while the **test** error drops and then rises. That rise is overfitting.

In [ ]:
depths = [1, 2, 3, 5, 8]
rows = []
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.6), sharey=True)
curve_x = np.linspace(0, 10, 200).reshape(-1, 1)
for ax, d in zip(axes, [1, 3, 8]):
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    ax.scatter(X_train[:, 0], y_train, c=TEAL, s=28, zorder=3)
    ax.scatter(X_test[:, 0], y_test, c=CORAL, s=28, marker="D", zorder=3)
    ax.plot(curve_x, m.predict(curve_x), color=INK, linewidth=2)
    ax.set_title(f"Depth {d}")
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    rows.append((d,
                 round(mean_absolute_error(y_train, m.predict(X_train)), 3),
                 round(mean_absolute_error(y_test, m.predict(X_test)), 3)))
print(pd.DataFrame(rows, columns=["max_depth", "train_MAE", "test_MAE"]).to_string(index=False))
plt.show()

**Regularisation** holds a flexible model back, so it cannot wiggle quite so wildly. `Ridge` is linear regression with a penalty on large coefficients. We will also **scale** the polynomial features, which keeps the maths stable.

**Cross-validation** is a more honest score than one lucky split. We rotate the test slice several times and average the results.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

wild = make_pipeline(PolynomialFeatures(12, include_bias=False), LinearRegression())
tame = make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), Ridge(alpha=0.01))
wild.fit(X_train, y_train)
tame.fit(X_train, y_train)
print(f"{'model':<28} {'train MAE':>10} {'test MAE':>10}")
print(f"{'Polynomial, no penalty':<28} {mean_absolute_error(y_train, wild.predict(X_train)):10.3f} {mean_absolute_error(y_test, wild.predict(X_test)):10.3f}")
print(f"{'Polynomial + Ridge':<28} {mean_absolute_error(y_train, tame.predict(X_train)):10.3f} {mean_absolute_error(y_test, tame.predict(X_test)):10.3f}")

cv = cross_val_score(LinearRegression(), houses[["size_sqm", "bedrooms"]], houses["price_k"], cv=5, scoring="r2")
print()
print("House-price R² on 5 folds:", np.round(cv, 3))
print("Mean R²:                  ", round(cv.mean(), 3))
print("Spread (std):             ", round(cv.std(), 3))

## Part 10. Pipelines and GridSearch
A **pipeline** bundles the prep and the model so they always run in the same order. The scaler is fit on the training fold only, which avoids **leakage** (test information sneaking into training).

We will scale the loan features and then fit logistic regression.

In [ ]:
from sklearn.pipeline import Pipeline

X = loans[["income_k", "credit_score", "years_at_job"]]
y = loans["approved"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
pipe.fit(X_train, y_train)
print("Pipeline test accuracy:", round(pipe.score(X_test, y_test), 3))

**Hyperparameters** are knobs we choose before training, such as `C` in logistic regression (smaller `C` means stronger regularisation).

**GridSearchCV** tries each candidate, uses cross-validation, and keeps the best setting. Then we check that winner on the held-out test set, once.

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipe, param_grid={"model__C": [0.01, 0.1, 1, 10, 100]}, cv=5)
grid.fit(X_train, y_train)
results = pd.DataFrame(grid.cv_results_)[["param_model__C", "mean_test_score"]]
results.columns = ["C", "cv_accuracy"]
print(results.round(3).to_string(index=False))
print()
print("Best C:        ", grid.best_params_["model__C"])
print("Best CV score: ", round(grid.best_score_, 3))
print("Held-out test: ", round(grid.score(X_test, y_test), 3))

## Part 11. Unsupervised learning
Sometimes there is no label. **Clustering** groups similar rows. **PCA** compresses many numbers into a few new axes that keep most of the variation.

This shop has customers with an age, an annual spend, and a visit count. We will look for natural groups.

In [ ]:
customers = load_csv("customers.csv")
print(customers.describe().round(1).to_string())

fig, ax = plt.subplots()
ax.scatter(customers["annual_spend"], customers["store_visits"], c=TEAL, s=55, edgecolors="white")
ax.set_xlabel("Annual spend")
ax.set_ylabel("Store visits")
ax.set_title("Customers, before clustering")
plt.show()

**K-means** asks us how many groups we want (`k=3` here), then finds 3 centres and assigns each customer to the nearest one. After that, we read the group averages and give them plain-English names.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
customers["cluster"] = kmeans.fit_predict(customers[["age", "annual_spend", "store_visits"]])
print(customers.groupby("cluster")[["age", "annual_spend", "store_visits"]].mean().round(1))
print()
print("Group sizes:")
print(customers["cluster"].value_counts().sort_index().to_string())

fig, ax = plt.subplots()
colours = [TEAL, CORAL, PURPLE]
for i, col in enumerate(colours):
    part = customers[customers["cluster"] == i]
    ax.scatter(part["annual_spend"], part["store_visits"], c=col, s=60, label=f"group {i}", edgecolors="white")
ax.set_xlabel("Annual spend")
ax.set_ylabel("Store visits")
ax.set_title("Customers, after k-means")
ax.legend()
plt.show()

**PCA** (principal component analysis) rotates the data onto new axes, starting with the direction of greatest spread. With 3 features, 2 axes are enough to draw a picture — and here they keep almost all of the variation.

In [ ]:
from sklearn.decomposition import PCA

scaled_cust = StandardScaler().fit_transform(customers[["age", "annual_spend", "store_visits"]])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(scaled_cust)
print("Variance kept by each component:", np.round(pca.explained_variance_ratio_, 3))
print("Total kept:                     ", round(pca.explained_variance_ratio_.sum(), 3))

fig, ax = plt.subplots()
for i, col in enumerate(colours):
    mask = customers["cluster"] == i
    ax.scatter(coords[mask, 0], coords[mask, 1], c=col, s=60, label=f"group {i}", edgecolors="white")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.set_title("The same groups, drawn with PCA")
ax.legend()
plt.show()

## Part 12. A small neural network
A **neural network** stacks simple units (neurons) into layers. Each unit is a weighted sum plus a curve, a bit like logistic regression. Together, the layers can learn more wiggly patterns.

We will use scikit-learn's `MLPClassifier` on `load_digits`: 1,797 handwritten digits, each an 8×8 grid of pixels. This is another built-in teaching set.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
print("Images:", digits.data.shape[0], "  features (pixels) each:", digits.data.shape[1])

fig, axes = plt.subplots(2, 8, figsize=(10.5, 3.2))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(label), fontsize=10)
    ax.axis("off")
fig.suptitle("A few handwritten digits", y=1.05)
plt.show()

One hidden layer of 32 neurons is enough for this small dataset. After training, we check the test accuracy, then look at a few predictions — including any the network got wrong.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=42, stratify=digits.target
)
mlp = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
print("Training rounds:", mlp.n_iter_)
print("Train accuracy: ", round(mlp.score(X_train, y_train), 3))
print("Test accuracy:  ", round(mlp.score(X_test, y_test), 3))

pred = mlp.predict(X_test)
wrong = np.where(pred != y_test)[0]
print("Mistakes on the test set:", len(wrong), "of", len(y_test))

show = list(wrong[:4]) + list(np.where(pred == y_test)[0][:4])
fig, axes = plt.subplots(1, 8, figsize=(10.5, 2.2))
for ax, i in zip(axes, show):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ok = pred[i] == y_test[i]
    ax.set_title(f"p={pred[i]}  y={y_test[i]}", color=(TEAL if ok else CORAL), fontsize=9)
    ax.axis("off")
fig.suptitle("Predictions  (p = predicted, y = true)", y=1.08)
plt.show()

## Part 13. Saving your model, and next steps
Once you are happy with a model, save it. **joblib** writes the fitted object to a file. Later, you load it and call `predict` — no need to train again.

We will save the house-price model from Part 2.

In [ ]:
import joblib

joblib.dump(linreg, "house_price_model.joblib")
loaded = joblib.load("house_price_model.joblib")
fresh = pd.DataFrame({"size_sqm": [100], "bedrooms": [3]})
print("Saved to house_price_model.joblib")
print("Loaded model prediction:", round(loaded.predict(fresh)[0], 1), "thousand")
print("Same as the original?  ", np.allclose(loaded.predict(fresh), linreg.predict(fresh)))

## Recap
You covered a lot in one hour:

1. Features, labels, and a train/test split.
2. Linear regression, with MAE, RMSE and R².
3. Logistic regression, and probabilities.
4. KNN, Naive Bayes and SVM.
5. Decision trees and random forests.
6. Gradient boosting.
7. Missing values, scaling, encoding, and new features.
8. The confusion matrix, precision, recall, ROC and imbalanced data.
9. Overfitting, regularisation and cross-validation.
10. Pipelines and GridSearch.
11. K-means and PCA.
12. A small neural network on digits.
13. Saving a model with joblib.

**Where to go next.** Try a bigger, real table of your own. Keep the split honest. Watch precision and recall, not just accuracy. And when a model will affect people, ask who could be hurt if it is wrong.

The notebook and the data are linked in the description. Change the examples, break things on purpose, and run the cells again. That is how you learn.